In [1]:
!uv pip install langchain openai tiktoken rapidocr-onnxruntime python-dotenv langchain-community

Using Python 3.13.7 environment at: C:\MLOPs\Fully_deploable_RAG\.venv
Resolved 72 packages in 2.09s
Prepared 6 packages in 8.22s
Installed 10 packages in 5.47s
 + flatbuffers==25.12.19
 + mpmath==1.3.0
 + onnxruntime==1.24.3
 + opencv-python==4.13.0.92
 + pillow==12.1.1
 + protobuf==7.34.0
 + pyclipper==1.4.0
 + rapidocr-onnxruntime==1.4.4
 + shapely==2.1.2
 + sympy==1.14.0


In [21]:
import os
from dotenv import load_dotenv

load_dotenv()

os.environ["NEBIUS_API_KEY"]=os.getenv("NEBIUS_API_KEY")

Data Ingestion

In [8]:
from langchain_community.document_loaders import TextLoader

In [11]:
loader=TextLoader(r"C:\MLOPs\Fully_deploable_RAG\data\AI_agent.txt", encoding="utf-8")
documents=loader.load()

<>:1: SyntaxWarning: invalid escape sequence '\M'
<>:1: SyntaxWarning: invalid escape sequence '\M'
C:\Users\Kartikey Agarwal\AppData\Local\Temp\ipykernel_19184\2301316522.py:1: SyntaxWarning: invalid escape sequence '\M'
  loader=TextLoader("C:\MLOPs\Fully_deploable_RAG\data\AI_agent.txt", encoding="utf-8")


In [13]:
documents[0].page_content[:500]

'### The Importance of Artificial Intelligence in Modern Society\n\nArtificial Intelligence (AI) has become one of the most transformative technologies of the 21st century. It refers to the ability of machines and computer systems to perform tasks that typically require human intelligence, such as learning, reasoning, problem-solving, and decision-making. Over the past decade, AI has moved from research laboratories into everyday life, significantly influencing industries, economies, and the way pe'

In [14]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter=RecursiveCharacterTextSplitter(chunk_size=200,chunk_overlap=20)

In [15]:
text_chunks=text_splitter.split_documents(documents)

In [16]:
text_chunks

[Document(metadata={'source': 'C:\\MLOPs\\Fully_deploable_RAG\\data\\AI_agent.txt'}, page_content='### The Importance of Artificial Intelligence in Modern Society'),
 Document(metadata={'source': 'C:\\MLOPs\\Fully_deploable_RAG\\data\\AI_agent.txt'}, page_content='Artificial Intelligence (AI) has become one of the most transformative technologies of the 21st century. It refers to the ability of machines and computer systems to perform tasks that typically'),
 Document(metadata={'source': 'C:\\MLOPs\\Fully_deploable_RAG\\data\\AI_agent.txt'}, page_content='that typically require human intelligence, such as learning, reasoning, problem-solving, and decision-making. Over the past decade, AI has moved from research laboratories into everyday life,'),
 Document(metadata={'source': 'C:\\MLOPs\\Fully_deploable_RAG\\data\\AI_agent.txt'}, page_content='into everyday life, significantly influencing industries, economies, and the way people interact with technology.'),
 Document(metadata={'source

In [18]:
from langchain_nebius.embeddings import NebiusEmbeddings
from langchain_community.vectorstores import FAISS

In [22]:
embeddings=NebiusEmbeddings(model="Qwen/Qwen3-Embedding-8B")

In [24]:
vectorstores=FAISS.from_documents(text_chunks,embeddings)

In [41]:
retriever=vectorstores.as_retriever()

In [26]:
query="Can AI ever fully replace human decision-making?"
docs=vectorstores.similarity_search(query,k=4)

for i , doc in enumerate(docs):
    print(f"Document {i+1}:")
    print(doc.page_content)
    print("-"*50)

Document 1:
### The Importance of Artificial Intelligence in Modern Society
--------------------------------------------------
Document 2:
that typically require human intelligence, such as learning, reasoning, problem-solving, and decision-making. Over the past decade, AI has moved from research laboratories into everyday life,
--------------------------------------------------
Document 3:
However, the rapid growth of AI also raises concerns about ethics, job displacement, and data privacy. As machines become more capable, it is important for governments, organizations, and researchers
--------------------------------------------------
Document 4:
In conclusion, artificial intelligence is reshaping the modern world by improving efficiency, enhancing decision-making, and enabling new technological possibilities. While challenges exist,
--------------------------------------------------


In [27]:
from langchain_core.prompts import ChatPromptTemplate

template="""
    You are an assistant for question-answering tasks.
    Use the following pieces of retriened context to answer the questions.
    If you don't know the answer , just say that you don't know.
    Use ten sentences maximum and keep the answer concise.

    Question: {question}
    Context: {context}
    Answer:
"""



In [28]:
prompt = ChatPromptTemplate.from_template(template)

In [31]:
prompt

ChatPromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, template="\n    You are an assistant for question-answering tasks.\n    Use the following pieces of retriened context to answer the questions.\n    If you don't know the answer , just say that you don't know.\n    Use ten sentences maximum and keep the answer concise.\n\n    Question: {question}\n    Context: {context}\n    Answer:\n"), additional_kwargs={})])

In [39]:
from langchain_core.output_parsers import StrOutputParser
from langchain_nebius import ChatNebius

In [37]:
output_parser=StrOutputParser()

In [40]:
llm_model=ChatNebius(model="openai/gpt-oss-20b")

In [43]:
from langchain_core.runnables import RunnablePassthrough
rag_Chain=(
    {"context":retriever , "question": RunnablePassthrough()}
    | prompt
    | llm_model
    | output_parser
)

In [44]:
rag_Chain.invoke("can AI replace human?")

'AI can automate many tasks that traditionally required human effort—especially those that rely on data analysis, pattern recognition, and routine decision‑making. Modern AI systems, as noted in the context, perform learning, reasoning, and problem‑solving at scale, which has transformed fields such as healthcare. However, AI generally lacks the breadth of human experience, creativity, and emotional intelligence that underpin many complex decisions. It also does not possess the same capacity for ethical judgment, moral responsibility, or social nuance. For tasks that require empathy, judgment in ambiguous situations, or physical presence, human involvement remains essential. Moreover, AI systems depend on human oversight to avoid biases, monitor performance, and provide contextual interpretation. Thus, while AI can replace humans in specific, well‑defined roles, it cannot fully supplant the human element across all domains.'